In [ ]:
import pandas as pd
import json
import math
import os
import requests
import re as _re
import numpy as np

from openai import OpenAI
from typing import Optional
from pydantic import BaseModel
from pydantic import Field

from shared import (
    build_course_desc,
    build_gened_pivot,
    build_clean_registration,
    build_major_courses_lookup,
    build_fy_details,
    build_course_title_info,
    student_profile_to_str,
)

from dotenv import load_dotenv
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=OPENAI_API_KEY)

In [3]:
# Clean up and build registration data 
# dfNew is the CRN dataframe which contains one row per CRN
# dfFinal is the deduplicated (no CRN)
dfNew, dfFinal = build_clean_registration("sample_fall2026_courses.csv")

In [4]:
# Build student profile string from survey response
# df = pd.read_excel('academic_advisor_survey.xlsx', sheet_name='Export')
df = pd.read_csv('sample_student_data.csv')

row = df.iloc[0]

student_profile_string = student_profile_to_str(row)

In [5]:
# Load data for AI recommendations
major_df = pd.read_excel("firstyearadvising.xlsx", sheet_name="majors_list")
fys_df = pd.read_excel("fys.xlsx")
gened_df = pd.read_excel("gened.xlsx")

# Major data — strip whitespace from names to avoid key mismatches (e.g. "Physics ")
major_df["major"] = major_df["major"].str.strip()
major_option_names = major_df["major"].tolist()
major_option_descriptions = major_df["level"].tolist()

# Build concentrations_lookup from the description column.
# Use re.split on the bullet character and strip both regular and non-breaking
# spaces (\xa0) that Excel sometimes inserts around bullet items.
import re as _re_conc
concentrations_lookup = {}
for _, row in major_df.iterrows():
    desc = str(row["description"]).strip()
    if desc.startswith("Possible concentrations offered:"):
        concentrations_lookup[str(row["major"]).strip()] = [
            _re_conc.sub(r"[\xa0\s]+", " ", c).strip()
            for c in _re_conc.split(r"[•]", desc)[1:]
            if c.strip()
        ]

# FYS course data
fys_option_names = fys_df["Course Name"].tolist()
fys_option_descriptions = fys_df["Description"].tolist()

# Gen-ed area data
gened_option_names = gened_df["Gen Ed Name"].tolist()
gened_option_descriptions = gened_df["Description"].tolist()

# Use cleaned-up registration data (dfFinal has ATTR 1 and ATTR 2 columns) for merging first-year list with full data
fy_details = build_fy_details("sample_fy_courses.xlsx", dfNew, dfFinal)
fy_details.to_excel("new.xlsx", index=False)

# First-year courses that fulfill at least one gen-ed
gened_courses_df = fy_details[
    (fy_details["ATTR 1"].notna() & (fy_details["ATTR 1"].str.strip() != "")) |
    (fy_details["ATTR 2"].notna() & (fy_details["ATTR 2"].str.strip() != ""))
].copy()

# Gen-Ed course data
gened_course_names = gened_courses_df["TITLE"].tolist()
gened_course_descriptions = gened_courses_df["COURSE_DESC"].fillna("").tolist()

In [6]:
major_courses_df = pd.read_excel("firstyearadvising.xlsx", sheet_name="major_entry_courses")

# Build a lookup: { "Anthropology": [ {course info}, ... ], ... }
major_courses_lookup = build_major_courses_lookup(major_courses_df)

In [7]:
# ── Lookup dicts from already-loaded data ────────────────────────────────────

fys_desc_lookup   = dict(zip(fys_df["Course Name"], fys_df["Description"]))
gened_desc_lookup = dict(zip(gened_df["Gen Ed Name"], gened_df["Description"]))

# course title (uppercase) → {desc, geneds}  — used by serialize_gened_course
course_title_to_info = build_course_title_info(fy_details)

# ── Build FY course catalog ───────────────────────────────────────────────────

fy_catalog = []

for _, row in fy_details.iterrows():
    desc   = row["COURSE_DESC"] if pd.notna(row["COURSE_DESC"]) else ""
    geneds = [g for g in [
        str(row["ATTR 1"]).strip() if pd.notna(row["ATTR 1"]) else "",
        str(row["ATTR 2"]).strip() if pd.notna(row["ATTR 2"]) else "",
    ] if g]

    reg_match = dfNew[
        (dfNew["SUBJ"] == row["SUBJ"]) &
        (dfNew["CRSE"].astype(str) == str(row["CRSE"])) &
        (dfNew["TITLE"] == row["TITLE"])
    ]
    crn_val = str(int(float(reg_match.iloc[0]["CRN"]))) if not reg_match.empty else None

    fy_catalog.append({
        "crn":    crn_val,
        "subj":   row["SUBJ"],
        "crse":   str(row["CRSE"]),
        "title":  row["TITLE"],
        "desc":   desc,
        "geneds": geneds,
    })

In [8]:
def final_recommend(
    client,
    model,
    student_profile_string,
    option_names_list,
    option_descriptions_list,
    option_category_description,
    number_of_options_to_recommend,
    additional_rules="",
    concentrations_lookup=None,
):
    
    # GPT-5 Nano pricing (update if pricing changes)
    INPUT_COST_PER_MILLION = 0.05
    OUTPUT_COST_PER_MILLION = 0.40

    # System prompt
    system_prompt = f"""
You are an academic advising assistant supporting a university professor.

You will be provided with:

1. A student profile containing the student's academic interests, hobbies, goals, and preferences.
2. A numbered list of available {option_category_description}.

Your task is to recommend {number_of_options_to_recommend} strong candidate options that may be valuable for the advisor to discuss with the student.

**PII Definition (apply throughout):**
Personally identifiable information (PII) includes: name, gender, pronouns, race or ethnicity, nationality, religion, specific employer or workplace details, or any other information that could identify a specific individual. Academic interests, hobbies, and general goals are NOT considered PII and may be freely referenced.

Rules:
- Only recommend options that appear in the provided list. Do not invent options.
- Base recommendations solely on the student's academic interests, hobbies, goals, and preferences.
- Never reference or repeat any PII (as defined above) anywhere in your output, including in the thought process section.
- Before making a recommendation, think step by step about how the student's interests and preferences connect to the available options.
- For each recommended option, provide 1, 2, or 3 concise sentences explaining why it is a strong fit.
- Provide only as many reasons as there are strong, distinct connections. Do not force 3 reasons if only 1 or 2 are meaningful.
- Reasons should be distinct from each other and should not repeat the same idea.
- Each reason should be specific to the student's profile but must not reference any PII (as defined above).{f"\n{additional_rules.strip()}" if additional_rules.strip() else ""}

Output format:
- A thought process section explaining the reasoning behind the recommendations, based only on the student's academic interests, hobbies, goals, and preferences. No PII.
- A list of the final recommended options, where each option includes:
  - option_number: the number corresponding to the recommended option from the provided list
  - option_name: the name of the recommended option exactly as it appears in the provided list
  - rank: the rank of this recommendation among all recommended options, where 1 = strongest fit
  - reasons: a paragraph of 1, 2, or 3 concise sentences explaining the reason, based on the student's interests, hobbies, goals, or preferences. No PII.
  """
    
    # Creating input options
    options_string = "\n".join([
        f"""Option #{i+1}\nOption Name: {name}\nOption Description: {description}\n"""
        for i, name, description in zip(range(len(option_names_list)), option_names_list, option_descriptions_list)
    ])

    input_prompt = f"""Student Profile:\n {student_profile_string}\n\nAvailable {option_category_description}: {options_string}"""

    # Output schema
    class option_choice(BaseModel):
        _conc_schema_note = (
            " IMPORTANT: If this option is one of the following programs that offer concentrations, "
            "you MUST name the specific concentration(s) that best fit the student's interests in this field: "
            + "; ".join(
                f"{prog} (concentrations: {', '.join(concs)})"
                for prog, concs in (concentrations_lookup or {}).items()
            )
            + ". Only mention concentrations that genuinely connect to the student's profile."
        ) if concentrations_lookup else ""
        reasons: str = Field(
            description=(
                "A description of 1, 2, or 3 concise sentences explaining the reason for why this option was selected, "
                "based on the student's profile. Reasons should be written without referencing any personally identifiable "
                "information (PII) from the profile. PII refers to any information that could potentially identify a specific "
                "individual, such as name, gender, race/ ethnicity, specific employment details. Other information such as "
                "academic interests, other interests, hobbies, and general goals can be used and referenced in the reasoning."
                + _conc_schema_note
            )
        )
        rank: int = Field(
            description="Rank of this recommendation among all recommended options, where 1 = strongest fit."
        )
        option_number: int = Field(
            description=f"The number corresponding to the recommended option from the provided list of {option_category_description}."
        )
        option_name: str = Field(
            description=f"The name of the recommended option, exactly as it appears in the provided list of {option_category_description}."
        )
    
    class recommendations(BaseModel):
        thought_process: str = Field(
            description="An explanation of the reasoning behind the recommendations, including how the student's interests and preferences were considered."
        )
        recommended_options: list[option_choice] = Field(
            description="A list of the final recommended options."
        )
    
    # Generate response 
    response = client.responses.parse(
        model=model,
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": input_prompt},
        ],
        text_format=recommendations,
    )

    input_tokens = response.usage.input_tokens
    output_tokens = response.usage.output_tokens

    estimated_cost = (
        (input_tokens * INPUT_COST_PER_MILLION / 1_000_000) 
        + (output_tokens * OUTPUT_COST_PER_MILLION / 1_000_000)
    )

    final_rec = response.output_parsed

    return final_rec, estimated_cost

In [9]:
# ── Student record builder ────────────────────────────────────────────

def build_student_record(row, client, model="gpt-5.4-nano"):
    profile_str = student_profile_to_str(row)

    _conc_lines = "\n".join(
        f"  * {program}: {', '.join(concentrations)}"
        for program, concentrations in concentrations_lookup.items()
    )
    _MAJOR_EXTRA = (
        "- Several programs offer concentrations. These concentrations are NOT listed separately "
        "in the options — recommend the parent program only. The programs with concentrations, "
        "and their available concentrations, are:\n"
        + _conc_lines + "\n"
        "- You MUST NOT recommend any individual concentration as a separate entry. Only recommend "
        "the parent program name exactly as it appears in the provided list.\n"
        "- When recommending a program that has concentrations (listed above), you MUST identify "
        "which specific concentration(s) from that program's list best match the student's interests "
        "and explicitly name them in your reasons. For example: 'Within this program, the Marketing "
        "concentration aligns well with the student's interest in...' Only suggest concentrations "
        "that genuinely connect to the student's profile — do not list all of them.\n"
        "- You MUST recommend exactly 10 options. Even if some fits are weaker, still "
        "include options ranked 8, 9, and 10 as lower-confidence suggestions rather than "
        "returning fewer than 10.\n"
        "- Rank the 10 recommendations from strongest fit (rank 1) to weakest fit (rank 10), "
        "and list them in that order."
    )

    major_recs,  major_cost  = final_recommend(client, model, profile_str, major_option_names, major_option_descriptions, "Academic Programs, including Majors, Minors, and Concentrations", 10, _MAJOR_EXTRA, concentrations_lookup=concentrations_lookup)
    fys_recs,    fys_cost    = final_recommend(client, model, profile_str, fys_option_names,    fys_option_descriptions,    "First-Year Seminar Courses", 25)
    gened_recs,  gened_cost  = final_recommend(client, model, profile_str, gened_option_names,  gened_option_descriptions,  "General Education Requirements", 5)
    course_recs, course_cost = final_recommend(client, model, profile_str, gened_course_names,  gened_course_descriptions,  "Gen-Ed Fulfilling Courses Available to First-Year Students", 20)

    def serialize_major(opt):
        lookup = major_courses_lookup.get(opt.option_name.strip(), {})
        return {
            "option_name":      opt.option_name,
            "option_number":    opt.option_number,
            "rank":             opt.rank,
            "reasons":          opt.reasons,
            "level":            lookup.get("level"),
            "additional_notes": lookup.get("additional_notes"),
            "courses":          lookup.get("courses", []),
            "concentrations":   list(concentrations_lookup.get(opt.option_name.strip(), [])),
        }

    def serialize_fys(opt):
        return {
            "option_name":   opt.option_name,
            "option_number": opt.option_number,
            "reasons":       opt.reasons,
            "description":   fys_desc_lookup.get(opt.option_name, ""),
        }

    def serialize_gened_area(opt):
        return {
            "option_name":   opt.option_name,
            "option_number": opt.option_number,
            "reasons":       opt.reasons,
            "description":   gened_desc_lookup.get(opt.option_name, ""),
        }

    def serialize_gened_course(opt):
        info = course_title_to_info.get(opt.option_name.upper().strip(), {})
        return {
            "option_name":   opt.option_name,
            "option_number": opt.option_number,
            "reasons":       opt.reasons,
            "description":   info.get("desc", ""),
            "geneds":        info.get("geneds", []),
        }

    return {
        "profile": {
            "academicInterests": [row["Academic Interest 1"], row["Academic Interest 2"]],
            "topInterests":      [row[f"Interest {i}"] for i in range(1, 6)],
            "recreation":        row["Recreation Hobbies"],
            "employment":        row["Employment Service"],
        },
        "recommendations": {
            "majors":       [serialize_major(opt)        for opt in major_recs.recommended_options],
            "fys":          [serialize_fys(opt)          for opt in fys_recs.recommended_options],
            "genedAreas":   [serialize_gened_area(opt)   for opt in gened_recs.recommended_options],
            "genedCourses": [serialize_gened_course(opt) for opt in course_recs.recommended_options],
        },
        "cost": round(major_cost + fys_cost + gened_cost + course_cost, 6),
    }

In [10]:
# Only run this once
# Import the packages

from __future__ import annotations
import uuid
from typing import Any
from sqlalchemy import String, create_engine
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column
from sqlalchemy.types import JSON

 
# Database File
DB_URL = "sqlite:///student_rec_data.db"
 
#
# Define and Create the database
#
class Base(DeclarativeBase):
    pass


class StudentAdvisingRec(Base):
    __tablename__ = "student_advising_rec"

    id: Mapped[str] = mapped_column(
        String,
        primary_key=True,
        default=lambda: str(uuid.uuid4()),
    )

    raw_id: Mapped[str] = mapped_column(
        String,
        nullable=False,
    )

    data: Mapped[dict[str, Any]] = mapped_column(
        JSON,
        nullable=False,
    )

engine = create_engine(DB_URL, echo=False)
Base.metadata.create_all(engine)

In [11]:
#
# Functions to add and get items
#

def add_item(raw_id: str, data: dict[str, Any]) -> str:

    item = StudentAdvisingRec(raw_id=raw_id, data=data)
    with Session(engine) as session:

        session.add(item)
        session.commit()
        session.refresh(item)
        return item.id

 

def get_item(item_id: str) -> dict[str, Any] | None:
    with Session(engine) as session:
        item = session.get(StudentAdvisingRec, item_id)

        if item is None:
            return None

        return {
            "id": item.id,
            "raw_id": item.raw_id,
            "data": item.data,
        }

In [12]:
# ── Full rebuild — USE WITH CAUTION ──────────────────────────────────────────
# Reprocesses every student from scratch: makes API calls for all rows,
# overwrites student_store.json entirely.
# Only run this if the store is lost or corrupted.
# For new students, run the incremental cell below instead.

student_store = {}

for i, (_, row) in enumerate(df.iterrows()):
    print(i,end=" ")
    student_id = row["RAW_ID"]
    record = build_student_record(row, client)
    record["student_id"] = student_id
    
    generated_id = add_item(
        raw_id=student_id,
        data=record
    )

student_store["_catalog"] = {"fy_courses": fy_catalog}

with open("project/student_store.json", "w") as f:
    json.dump(student_store, f, indent=2)

0 1 2 3 4 

In [13]:
import sqlite3
import pandas as pd

# Connect to the SQLite database
conn = sqlite3.connect("student_rec_data.db")

# --- Option 1: Load a specific table ---
df = pd.read_sql_query("SELECT * FROM student_advising_rec", conn)
df.to_csv("database_to_csv.csv", index=False)

In [ ]:
# Load existing store so we don't reprocess anyone
if os.path.exists("project/student_store.json"):
    with open("project/student_store.json") as f:
        student_store = json.load(f)
else:
    student_store = {}

# Process new students (makes API calls only for students not yet in the store)
for i, (_, row) in enumerate(df.iterrows()):
    student_id = str(i)
    if student_id in student_store:
        continue
    record = build_student_record(row, client)
    record["student_id"] = student_id
    student_store[student_id] = record

# Enrich existing records with descriptions (no API calls — just Excel lookups)
for student_id, record in student_store.items():
    if student_id.startswith("_"):
        continue
    recs = record.get("recommendations", {})
    for opt in recs.get("fys", []):
        if "description" not in opt:
            opt["description"] = fys_desc_lookup.get(opt["option_name"], "")
    for opt in recs.get("genedAreas", []):
        if "description" not in opt:
            opt["description"] = gened_desc_lookup.get(opt["option_name"], "")
    for opt in recs.get("genedCourses", []):
        if "description" not in opt:
            info = course_title_to_info.get(opt["option_name"].upper().strip(), {})
            opt["description"] = info.get("desc", "")
            opt["geneds"]      = info.get("geneds", [])

# Update catalog (always regenerated from current Excel data)
student_store["_catalog"] = {"fy_courses": fy_catalog}

# Save
with open("project/student_store.json", "w") as f:
    json.dump(student_store, f, indent=2)

print(f"Done. {len([k for k in student_store if not k.startswith('_')])} total students saved.")

# Tell the running server to reload — no restart needed
try:
    requests.post("http://localhost:8000/api/reload", json={"secret": "your-secret-here"})
    print("Server reloaded.")
except:
    print("Server not running or reload failed — start the server if needed.")
